# 05b — Feature Engineering: Dataset con Variables Categóricas (para CatBoost)

Extiende el dataset de `features_train.parquet` (30 features numéricas) agregando variables categóricas de `application_train/test` en formato string, para ser usadas por CatBoost de forma nativa (target encoding ordenado, sin encoding manual).

**Input:**
- `data/processed/features_train.parquet` — 30 features numéricas + SK_ID_CURR + TARGET
- `data/processed/features_test.parquet` — 30 features numéricas + SK_ID_CURR
- `data/processed/app_train.parquet` — tabla de aplicaciones (columnas categóricas originales)
- `data/processed/app_test.parquet` — idem para test

**Output:**
- `data/processed/features_train_cat.parquet` — 36 features (30 num + 6 cat) + SK_ID_CURR + TARGET
- `data/processed/features_test_cat.parquet` — 36 features + SK_ID_CURR

**Columnas categóricas agregadas:**
- `NAME_CONTRACT_TYPE` — 2 valores (Cash loans, Revolving loans)
- `NAME_INCOME_TYPE` — 8 valores (Working, Pensioner, Commercial associate, etc.)
- `NAME_EDUCATION_TYPE` — 5 valores
- `NAME_FAMILY_STATUS` — 5 valores
- `NAME_HOUSING_TYPE` — 6 valores
- `OCCUPATION_TYPE` — 18 valores + ~30% NaN (CatBoost maneja NaN nativamente)

**Nota:** `CODE_GENDER` ya está en los 30 features como numérico (0/1). Se mantiene así para compatibilidad con los demás modelos.

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

ROOT      = os.path.abspath(os.path.join('..', '..'))
DATA_PROC = os.path.join(ROOT, 'data', 'processed')

print(f'ROOT: {ROOT}')
print(f'DATA_PROC: {DATA_PROC}')

# Columnas categóricas a agregar desde application_train/test
CAT_COLS = [
    'NAME_CONTRACT_TYPE',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
]

ROOT: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07
DATA_PROC: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\data\processed


In [2]:
# ─── Cargar los parquets de features existentes ───────────────────────────────
print('Cargando features existentes...')
df_train = pd.read_parquet(os.path.join(DATA_PROC, 'features_train.parquet'))
df_test  = pd.read_parquet(os.path.join(DATA_PROC, 'features_test.parquet'))

print(f'  features_train.parquet : {df_train.shape}')
print(f'  features_test.parquet  : {df_test.shape}')
print(f'  Columnas: {[c for c in df_train.columns if c not in ("SK_ID_CURR", "TARGET")]}')

Cargando features existentes...
  features_train.parquet : (307511, 32)
  features_test.parquet  : (48744, 31)
  Columnas: ['CREDIT_ANNUITY_RATIO', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'INSTAL_PAYMENT_DIFF_MEAN', 'DAYS_EMPLOYED_PERCENT', 'PREV_CNT_PAYMENT_MEAN', 'PREV_AMT_ANNUITY_MEAN', 'ANNUITY_INCOME_RATIO', 'POS_MONTHS_COUNT', 'BURO_AMT_CREDIT_SUM_SUM', 'CREDIT_GOODS_RATIO', 'PREV_DAYS_DECISION_MEAN', 'PREV_APP_CREDIT_RATIO_MEAN', 'BURO_DEBT_CREDIT_RATIO', 'CC_UTILIZATION_MEAN', 'DAYS_ID_PUBLISH', 'PREV_REFUSED_RATIO', 'BURO_DAYS_CREDIT_MEAN', 'BURO_CREDIT_ACTIVE_COUNT', 'DAYS_REGISTRATION', 'CC_AMT_BALANCE_MEAN', 'PREV_AMT_CREDIT_MEAN', 'OWN_CAR_AGE', 'BURO_BB_DPD_MEAN_MEAN', 'BURO_DAYS_CREDIT_MIN', 'CODE_GENDER', 'PREV_APPROVED_RATIO', 'BURO_AMT_CREDIT_SUM_OVERDUE_SUM']


In [3]:
# ─── Cargar columnas categóricas desde application_train/test ────────────────
print('Cargando application parquets...')
app_train = pd.read_parquet(os.path.join(DATA_PROC, 'app_train.parquet'),
                             columns=['SK_ID_CURR'] + CAT_COLS)
app_test  = pd.read_parquet(os.path.join(DATA_PROC, 'app_test.parquet'),
                             columns=['SK_ID_CURR'] + CAT_COLS)

print(f'  app_train shape: {app_train.shape}')
print(f'  app_test  shape: {app_test.shape}')

# Inspección de valores únicos y NaN por columna
print('\nColumnas categóricas — valores únicos y % NaN:')
for col in CAT_COLS:
    n_unique = app_train[col].nunique()
    na_pct   = app_train[col].isnull().mean() * 100
    valores  = sorted(app_train[col].dropna().unique().tolist())
    print(f'  {col:<25s}: {n_unique:>2} únicos, {na_pct:5.1f}% NaN')
    print(f'    Valores: {valores}')

Cargando application parquets...
  app_train shape: (307511, 7)
  app_test  shape: (48744, 7)

Columnas categóricas — valores únicos y % NaN:
  NAME_CONTRACT_TYPE       :  2 únicos,   0.0% NaN
    Valores: ['Cash loans', 'Revolving loans']
  NAME_INCOME_TYPE         :  8 únicos,   0.0% NaN
    Valores: ['Businessman', 'Commercial associate', 'Maternity leave', 'Pensioner', 'State servant', 'Student', 'Unemployed', 'Working']
  NAME_EDUCATION_TYPE      :  5 únicos,   0.0% NaN
    Valores: ['Academic degree', 'Higher education', 'Incomplete higher', 'Lower secondary', 'Secondary / secondary special']
  NAME_FAMILY_STATUS       :  6 únicos,   0.0% NaN
    Valores: ['Civil marriage', 'Married', 'Separated', 'Single / not married', 'Unknown', 'Widow']
  NAME_HOUSING_TYPE        :  6 únicos,   0.0% NaN
    Valores: ['Co-op apartment', 'House / apartment', 'Municipal apartment', 'Office apartment', 'Rented apartment', 'With parents']
  OCCUPATION_TYPE          : 18 únicos,  31.3% NaN
    Valo

In [4]:
# ─── Merge: agregar categóricas al dataset de features ───────────────────────
print('Mergeando columnas categóricas...')

df_train_cat = df_train.merge(app_train, on='SK_ID_CURR', how='left')
df_test_cat  = df_test.merge(app_test,   on='SK_ID_CURR', how='left')

# Verificar shapes
print(f'  Train con categóricas: {df_train_cat.shape}  (antes: {df_train.shape})')
print(f'  Test  con categóricas: {df_test_cat.shape}  (antes: {df_test.shape})')

# Verificar que no se perdieron filas
assert len(df_train_cat) == len(df_train), 'ERROR: se perdieron filas en train!'
assert len(df_test_cat)  == len(df_test),  'ERROR: se perdieron filas en test!'

# Verificar dtypes de las nuevas columnas
print('\nDtypes de columnas categóricas (deben ser object):')
for col in CAT_COLS:
    print(f'  {col:<25s}: {df_train_cat[col].dtype}')

Mergeando columnas categóricas...
  Train con categóricas: (307511, 38)  (antes: (307511, 32))
  Test  con categóricas: (48744, 37)  (antes: (48744, 31))

Dtypes de columnas categóricas (deben ser object):
  NAME_CONTRACT_TYPE       : object
  NAME_INCOME_TYPE         : object
  NAME_EDUCATION_TYPE      : object
  NAME_FAMILY_STATUS       : object
  NAME_HOUSING_TYPE        : object
  OCCUPATION_TYPE          : object


In [5]:
# ─── Resumen del dataset final ────────────────────────────────────────────────
feature_cols = [c for c in df_train_cat.columns if c not in ('SK_ID_CURR', 'TARGET')]
num_cols = [c for c in feature_cols if df_train_cat[c].dtype != 'object']
cat_cols = [c for c in feature_cols if df_train_cat[c].dtype == 'object']

print('=' * 65)
print('DATASET CON CATEGÓRICAS — RESUMEN')
print('=' * 65)
print(f'  Total features    : {len(feature_cols)}  ({len(num_cols)} numéricas + {len(cat_cols)} categóricas)')
print(f'  SK_ID_CURR        : 1 columna de ID')
print(f'  TARGET            : 1 columna (solo en train)')
print(f'  Filas train       : {len(df_train_cat):,}')
print(f'  Filas test        : {len(df_test_cat):,}')
print('=' * 65)
print(f'\nColumnas numéricas ({len(num_cols)}):')
for c in num_cols:
    na_pct = df_train_cat[c].isnull().mean() * 100
    print(f'  {c:<40s} NA={na_pct:5.1f}%')
print(f'\nColumnas categóricas ({len(cat_cols)}):')
for c in cat_cols:
    na_pct = df_train_cat[c].isnull().mean() * 100
    print(f'  {c:<40s} NA={na_pct:5.1f}%')

DATASET CON CATEGÓRICAS — RESUMEN
  Total features    : 36  (30 numéricas + 6 categóricas)
  SK_ID_CURR        : 1 columna de ID
  TARGET            : 1 columna (solo en train)
  Filas train       : 307,511
  Filas test        : 48,744

Columnas numéricas (30):
  CREDIT_ANNUITY_RATIO                     NA=  0.0%
  EXT_SOURCE_1                             NA= 56.4%
  EXT_SOURCE_2                             NA=  0.2%
  EXT_SOURCE_3                             NA= 19.8%
  DAYS_BIRTH                               NA=  0.0%
  INSTAL_PAYMENT_DIFF_MEAN                 NA=  5.2%
  DAYS_EMPLOYED_PERCENT                    NA= 18.0%
  PREV_CNT_PAYMENT_MEAN                    NA=  5.5%
  PREV_AMT_ANNUITY_MEAN                    NA=  5.5%
  ANNUITY_INCOME_RATIO                     NA=  0.0%
  POS_MONTHS_COUNT                         NA=  5.9%
  BURO_AMT_CREDIT_SUM_SUM                  NA= 14.3%
  CREDIT_GOODS_RATIO                       NA=  0.1%
  PREV_DAYS_DECISION_MEAN                  NA=  5

In [6]:
# ─── Guardar parquets ─────────────────────────────────────────────────────────
TRAIN_OUT = os.path.join(DATA_PROC, 'features_train_cat.parquet')
TEST_OUT  = os.path.join(DATA_PROC, 'features_test_cat.parquet')

df_train_cat.to_parquet(TRAIN_OUT, index=False)
df_test_cat.to_parquet(TEST_OUT,   index=False)

print('ARTEFACTOS GUARDADOS')
print(f'  {TRAIN_OUT}')
print(f'    Shape: {df_train_cat.shape}  |  Tamaño: {os.path.getsize(TRAIN_OUT)/1e6:.1f} MB')
print(f'  {TEST_OUT}')
print(f'    Shape: {df_test_cat.shape}  |  Tamaño: {os.path.getsize(TEST_OUT)/1e6:.1f} MB')

ARTEFACTOS GUARDADOS
  c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\data\processed\features_train_cat.parquet
    Shape: (307511, 38)  |  Tamaño: 29.6 MB
  c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\data\processed\features_test_cat.parquet
    Shape: (48744, 37)  |  Tamaño: 5.7 MB


In [7]:
# ─── Verificación rápida de lectura ───────────────────────────────────────────
check_train = pd.read_parquet(TRAIN_OUT)
check_test  = pd.read_parquet(TEST_OUT)

print('Verificación:')
print(f'  features_train_cat.parquet: {check_train.shape} ✓')
print(f'  features_test_cat.parquet : {check_test.shape}  ✓')
print(f'\nPrimeras 3 filas (columnas categóricas):')
display(check_train[['SK_ID_CURR'] + CAT_COLS].head(3))

print('\nListos para usar en catboost2-cloud-train.ipynb')

Verificación:
  features_train_cat.parquet: (307511, 38) ✓
  features_test_cat.parquet : (48744, 37)  ✓

Primeras 3 filas (columnas categóricas):


,SK_ID_CURR,NAME_CONTRACT_TYPE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE
0,100002,Cash loans,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers
1,100003,Cash loans,State servant,Higher education,Married,House / apartment,Core staff
2,100004,Revolving loans,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers



Listos para usar en catboost2-cloud-train.ipynb
